In [4]:
from pathlib import Path

import joblib
import numpy as np
import pandas as pd

from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import (
    GridSearchCV,
    StratifiedKFold
)
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler

from sklearn.metrics import (
    accuracy_score,
    average_precision_score,
    balanced_accuracy_score,
    confusion_matrix,
    f1_score,
    precision_score,
    recall_score,
    roc_auc_score
)

RANDOM_STATE = 42

In [5]:
DATA_DIR = Path("data")

checkpoint_path = (
    DATA_DIR /
    "GSE25055_pre_lasso.joblib"
)

checkpoint = joblib.load(
    checkpoint_path
)

X = checkpoint["X"]
y = checkpoint["y"]

print("Dataset:", checkpoint["dataset_id"])
print("X shape:", X.shape)
print("y shape:", y.shape)

print("\nClass distribution:")
print(y.value_counts().sort_index())

Dataset: GSE25055
X shape: (306, 22283)
y shape: (306,)

Class distribution:
pCR
0    249
1     57
Name: count, dtype: int64


In [6]:
assert len(X) == len(y)
assert X.index.equals(y.index)
assert X.isna().sum().sum() == 0
assert y.isna().sum() == 0
assert set(y.unique()) == {0, 1}

print("X and y are valid and correctly aligned.")

X and y are valid and correctly aligned.


In [7]:
outer_cv = StratifiedKFold(
    n_splits=10,
    shuffle=True,
    random_state=RANDOM_STATE
)

inner_cv = StratifiedKFold(
    n_splits=5,
    shuffle=True,
    random_state=RANDOM_STATE
)

outer_splits = list(
    outer_cv.split(X, y)
)

print("Outer folds:", len(outer_splits))

for fold, (
    training_indices,
    validation_indices
) in enumerate(
    outer_splits,
    start=1
):
    print(
        f"Fold {fold}: "
        f"training={len(training_indices)}, "
        f"validation={len(validation_indices)}"
    )

Outer folds: 10
Fold 1: training=275, validation=31
Fold 2: training=275, validation=31
Fold 3: training=275, validation=31
Fold 4: training=275, validation=31
Fold 5: training=275, validation=31
Fold 6: training=275, validation=31
Fold 7: training=276, validation=30
Fold 8: training=276, validation=30
Fold 9: training=276, validation=30
Fold 10: training=276, validation=30


In [8]:
def create_l2_logistic_pipeline():
    return Pipeline([
        (
            "scaler",
            StandardScaler()
        ),
        (
            "logistic",
            LogisticRegression(
                penalty="l2",
                solver="liblinear",
                max_iter=10000,
                random_state=RANDOM_STATE
            )
        )
    ])


l2_parameter_grid = {
    "logistic__C": np.logspace(
        -4,
        2,
        9
    ),
    "logistic__class_weight": [
        None,
        "balanced"
    ]
}

print("L2 Logistic Regression pipeline:")
print(create_l2_logistic_pipeline())

print("\nValues of C:")
print(l2_parameter_grid["logistic__C"])

L2 Logistic Regression pipeline:
Pipeline(steps=[('scaler', StandardScaler()),
                ('logistic',
                 LogisticRegression(max_iter=10000, penalty='l2',
                                    random_state=42, solver='liblinear'))])

Values of C:
[1.00000000e-04 5.62341325e-04 3.16227766e-03 1.77827941e-02
 1.00000000e-01 5.62341325e-01 3.16227766e+00 1.77827941e+01
 1.00000000e+02]


In [9]:
l2_fold_records = []

l2_oof_probability = pd.Series(
    np.nan,
    index=y.index,
    name="pCR_probability"
)

l2_oof_prediction = pd.Series(
    pd.NA,
    index=y.index,
    dtype="Int64",
    name="predicted_response"
)

for fold, (
    training_indices,
    validation_indices
) in enumerate(
    outer_splits,
    start=1
):

    X_train = X.iloc[training_indices]
    X_validation = X.iloc[validation_indices]

    y_train = y.iloc[training_indices]
    y_validation = y.iloc[validation_indices]

    grid_search = GridSearchCV(
        estimator=create_l2_logistic_pipeline(),
        param_grid=l2_parameter_grid,
        scoring="average_precision",
        cv=inner_cv,
        refit=True,
        n_jobs=1
    )

    grid_search.fit(
        X_train,
        y_train
    )

    best_model = grid_search.best_estimator_

    validation_probability = (
        best_model.predict_proba(
            X_validation
        )[:, 1]
    )

    validation_prediction = (
        best_model.predict(
            X_validation
        )
    )

    # Save out-of-fold predictions
    l2_oof_probability.loc[
        X_validation.index
    ] = validation_probability

    l2_oof_prediction.loc[
        X_validation.index
    ] = validation_prediction

    tn, fp, fn, tp = confusion_matrix(
        y_validation,
        validation_prediction,
        labels=[0, 1]
    ).ravel()

    sensitivity = (
        tp / (tp + fn)
        if (tp + fn) > 0
        else np.nan
    )

    specificity = (
        tn / (tn + fp)
        if (tn + fp) > 0
        else np.nan
    )

    fold_result = {
        "fold": fold,
        "training_patients": len(training_indices),
        "validation_patients": len(validation_indices),

        "best_C": grid_search.best_params_[
            "logistic__C"
        ],

        "best_class_weight": grid_search.best_params_[
            "logistic__class_weight"
        ],

        "roc_auc": roc_auc_score(
            y_validation,
            validation_probability
        ),

        "average_precision": average_precision_score(
            y_validation,
            validation_probability
        ),

        "accuracy": accuracy_score(
            y_validation,
            validation_prediction
        ),

        "balanced_accuracy": balanced_accuracy_score(
            y_validation,
            validation_prediction
        ),

        "f1": f1_score(
            y_validation,
            validation_prediction,
            zero_division=0
        ),

        "precision": precision_score(
            y_validation,
            validation_prediction,
            zero_division=0
        ),

        "sensitivity": sensitivity,
        "specificity": specificity,

        "true_RD": int((y_validation == 0).sum()),
        "true_pCR": int((y_validation == 1).sum()),

        "predicted_RD": int(
            (validation_prediction == 0).sum()
        ),

        "predicted_pCR": int(
            (validation_prediction == 1).sum()
        )
    }

    l2_fold_records.append(
        fold_result
    )

    print(
        f"Fold {fold}/10 completed | "
        f"best C={fold_result['best_C']} | "
        f"class_weight="
        f"{fold_result['best_class_weight']} | "
        f"ROC-AUC={fold_result['roc_auc']:.3f} | "
        f"PR-AUC="
        f"{fold_result['average_precision']:.3f}"
    )

d:\diplom-project\.venv\Lib\site-packages\sklearn\linear_model\_logistic.py:1403: FutureWarning: 'penalty' was deprecated in version 1.8 and will be removed in 1.10. To avoid this warning, leave 'penalty' set to its default value and use 'l1_ratio' or 'C' instead. Use l1_ratio=0 instead of penalty='l2', l1_ratio=1 instead of penalty='l1', l1_ratio set to a float between 0 and 1 instead of penalty='elasticnet', and C=np.inf instead of penalty=None.
  warnings.warn(
d:\diplom-project\.venv\Lib\site-packages\sklearn\linear_model\_logistic.py:1403: FutureWarning: 'penalty' was deprecated in version 1.8 and will be removed in 1.10. To avoid this warning, leave 'penalty' set to its default value and use 'l1_ratio' or 'C' instead. Use l1_ratio=0 instead of penalty='l2', l1_ratio=1 instead of penalty='l1', l1_ratio set to a float between 0 and 1 instead of penalty='elasticnet', and C=np.inf instead of penalty=None.
  warnings.warn(
d:\diplom-project\.venv\Lib\site-packages\sklearn\linear_model

Fold 1/10 completed | best C=100.0 | class_weight=None | ROC-AUC=0.860 | PR-AUC=0.604


d:\diplom-project\.venv\Lib\site-packages\sklearn\linear_model\_logistic.py:1403: FutureWarning: 'penalty' was deprecated in version 1.8 and will be removed in 1.10. To avoid this warning, leave 'penalty' set to its default value and use 'l1_ratio' or 'C' instead. Use l1_ratio=0 instead of penalty='l2', l1_ratio=1 instead of penalty='l1', l1_ratio set to a float between 0 and 1 instead of penalty='elasticnet', and C=np.inf instead of penalty=None.
  warnings.warn(
d:\diplom-project\.venv\Lib\site-packages\sklearn\linear_model\_logistic.py:1403: FutureWarning: 'penalty' was deprecated in version 1.8 and will be removed in 1.10. To avoid this warning, leave 'penalty' set to its default value and use 'l1_ratio' or 'C' instead. Use l1_ratio=0 instead of penalty='l2', l1_ratio=1 instead of penalty='l1', l1_ratio set to a float between 0 and 1 instead of penalty='elasticnet', and C=np.inf instead of penalty=None.
  warnings.warn(
d:\diplom-project\.venv\Lib\site-packages\sklearn\linear_model

Fold 2/10 completed | best C=0.0001 | class_weight=balanced | ROC-AUC=0.773 | PR-AUC=0.546


d:\diplom-project\.venv\Lib\site-packages\sklearn\linear_model\_logistic.py:1403: FutureWarning: 'penalty' was deprecated in version 1.8 and will be removed in 1.10. To avoid this warning, leave 'penalty' set to its default value and use 'l1_ratio' or 'C' instead. Use l1_ratio=0 instead of penalty='l2', l1_ratio=1 instead of penalty='l1', l1_ratio set to a float between 0 and 1 instead of penalty='elasticnet', and C=np.inf instead of penalty=None.
  warnings.warn(
d:\diplom-project\.venv\Lib\site-packages\sklearn\linear_model\_logistic.py:1403: FutureWarning: 'penalty' was deprecated in version 1.8 and will be removed in 1.10. To avoid this warning, leave 'penalty' set to its default value and use 'l1_ratio' or 'C' instead. Use l1_ratio=0 instead of penalty='l2', l1_ratio=1 instead of penalty='l1', l1_ratio set to a float between 0 and 1 instead of penalty='elasticnet', and C=np.inf instead of penalty=None.
  warnings.warn(
d:\diplom-project\.venv\Lib\site-packages\sklearn\linear_model

...

Fold 8/10 completed | best C=0.0001 | class_weight=balanced | ROC-AUC=0.704 | PR-AUC=0.479


d:\diplom-project\.venv\Lib\site-packages\sklearn\linear_model\_logistic.py:1403: FutureWarning: 'penalty' was deprecated in version 1.8 and will be removed in 1.10. To avoid this warning, leave 'penalty' set to its default value and use 'l1_ratio' or 'C' instead. Use l1_ratio=0 instead of penalty='l2', l1_ratio=1 instead of penalty='l1', l1_ratio set to a float between 0 and 1 instead of penalty='elasticnet', and C=np.inf instead of penalty=None.
  warnings.warn(
d:\diplom-project\.venv\Lib\site-packages\sklearn\linear_model\_logistic.py:1403: FutureWarning: 'penalty' was deprecated in version 1.8 and will be removed in 1.10. To avoid this warning, leave 'penalty' set to its default value and use 'l1_ratio' or 'C' instead. Use l1_ratio=0 instead of penalty='l2', l1_ratio=1 instead of penalty='l1', l1_ratio set to a float between 0 and 1 instead of penalty='elasticnet', and C=np.inf instead of penalty=None.
  warnings.warn(
d:\diplom-project\.venv\Lib\site-packages\sklearn\linear_model

Fold 9/10 completed | best C=0.0001 | class_weight=None | ROC-AUC=0.888 | PR-AUC=0.597


d:\diplom-project\.venv\Lib\site-packages\sklearn\linear_model\_logistic.py:1403: FutureWarning: 'penalty' was deprecated in version 1.8 and will be removed in 1.10. To avoid this warning, leave 'penalty' set to its default value and use 'l1_ratio' or 'C' instead. Use l1_ratio=0 instead of penalty='l2', l1_ratio=1 instead of penalty='l1', l1_ratio set to a float between 0 and 1 instead of penalty='elasticnet', and C=np.inf instead of penalty=None.
  warnings.warn(
d:\diplom-project\.venv\Lib\site-packages\sklearn\linear_model\_logistic.py:1403: FutureWarning: 'penalty' was deprecated in version 1.8 and will be removed in 1.10. To avoid this warning, leave 'penalty' set to its default value and use 'l1_ratio' or 'C' instead. Use l1_ratio=0 instead of penalty='l2', l1_ratio=1 instead of penalty='l1', l1_ratio set to a float between 0 and 1 instead of penalty='elasticnet', and C=np.inf instead of penalty=None.
  warnings.warn(
d:\diplom-project\.venv\Lib\site-packages\sklearn\linear_model

Fold 10/10 completed | best C=0.0005623413251903491 | class_weight=balanced | ROC-AUC=0.750 | PR-AUC=0.352


In [10]:
assert l2_oof_probability.notna().all()
assert l2_oof_prediction.notna().all()

l2_cv_results = pd.DataFrame(
    l2_fold_records
)

display(l2_cv_results)

,fold,training_patients,validation_patients,best_C,best_class_weight,roc_auc,average_precision,accuracy,balanced_accuracy,f1,precision,sensitivity,specificity,true_RD,true_pCR,predicted_RD,predicted_pCR
0,1,275,31,100.000000,NaN,0.860000,0.603535,0.741935,0.650000,0.428571,0.375000,0.500000,0.800000,25,6,23,8
1,2,275,31,0.000100,balanced,0.773333,0.545869,0.806452,0.816667,0.625000,0.500000,0.833333,0.800000,25,6,21,10
2,3,275,31,0.017783,balanced,0.673333,0.352486,0.677419,0.736667,0.500000,0.357143,0.833333,0.640000,25,6,17,14
3,4,275,31,0.000562,NaN,0.646667,0.416966,0.580645,0.613333,0.380952,0.266667,0.666667,0.560000,25,6,16,15
4,5,275,31,0.000562,balanced,0.926667,0.743849,0.612903,0.760000,0.500000,0.333333,1.000000,0.520000,25,6,13,18
5,6,275,31,0.000100,NaN,0.806667,0.542424,0.580645,0.676667,0.434783,0.294118,0.833333,0.520000,25,6,14,17
6,7,276,30,0.000562,balanced,0.912000,0.564444,0.700000,0.820000,0.526316,0.357143,1.000000,0.640000,25,5,16,14
7,8,276,30,0.000100,balanced,0.704000,0.478575,0.633333,0.620000,0.352941,0.250000,0.600000,0.640000,25,5,18,12
8,9,276,30,0.000100,NaN,0.888000,0.596825,0.866667,0.920000,0.714286,0.555556,1.000000,0.840000,25,5,21,9
9,10,276,30,0.000562,balanced,0.750000,0.352066,0.700000,0.750000,0.526316,0.384615,0.833333,0.666667,24,6,17,13


In [11]:
l2_metric_columns = [
    "roc_auc",
    "average_precision",
    "accuracy",
    "balanced_accuracy",
    "f1",
    "precision",
    "sensitivity",
    "specificity"
]

l2_cv_summary = (
    l2_cv_results[
        l2_metric_columns
    ]
    .agg([
        "mean",
        "std"
    ])
    .T
)

l2_cv_summary.columns = [
    "Mean",
    "Standard deviation"
]

print("L2 Logistic Regression CV summary:")

display(l2_cv_summary)

L2 Logistic Regression CV summary:


,Mean,Standard deviation
roc_auc,0.794067,0.100889
average_precision,0.519704,0.122352
accuracy,0.690000,0.094810
balanced_accuracy,0.736333,0.098663
f1,0.498916,0.109580
precision,0.367357,0.096615
sensitivity,0.810000,0.173597
specificity,0.662667,0.116416


In [12]:
l2_oof_prediction_int = (
    l2_oof_prediction
    .astype("int64")
)

l2_confusion_matrix = confusion_matrix(
    y,
    l2_oof_prediction_int,
    labels=[0, 1]
)

tn, fp, fn, tp = (
    l2_confusion_matrix.ravel()
)

l2_sensitivity = (
    tp / (tp + fn)
)

l2_specificity = (
    tn / (tn + fp)
)

l2_overall_metrics = pd.DataFrame({
    "Metric": [
        "ROC-AUC",
        "PR-AUC",
        "Accuracy",
        "Balanced accuracy",
        "F1",
        "Precision",
        "Sensitivity",
        "Specificity"
    ],
    "Value": [
        roc_auc_score(
            y,
            l2_oof_probability
        ),
        average_precision_score(
            y,
            l2_oof_probability
        ),
        accuracy_score(
            y,
            l2_oof_prediction_int
        ),
        balanced_accuracy_score(
            y,
            l2_oof_prediction_int
        ),
        f1_score(
            y,
            l2_oof_prediction_int,
            zero_division=0
        ),
        precision_score(
            y,
            l2_oof_prediction_int,
            zero_division=0
        ),
        l2_sensitivity,
        l2_specificity
    ]
})

print("L2 overall out-of-fold performance:")

display(l2_overall_metrics)

print("\nConfusion matrix:")
print(l2_confusion_matrix)

L2 overall out-of-fold performance:


,Metric,Value
0,ROC-AUC,0.770873
1,PR-AUC,0.414349
2,Accuracy,0.689542
3,Balanced accuracy,0.734834
4,F1,0.491979
5,Precision,0.353846
6,Sensitivity,0.807018
7,Specificity,0.662651



Confusion matrix:
[[165  84]
 [ 11  46]]


In [13]:
from pathlib import Path

RESULTS_DIR = Path("results")

RESULTS_DIR.mkdir(
    parents=True,
    exist_ok=True
)

l2_cv_results.to_csv(
    RESULTS_DIR /
    "l2_logistic_cv_fold_metrics.csv",
    index=False
)

l2_cv_summary.to_csv(
    RESULTS_DIR /
    "l2_logistic_cv_summary.csv"
)

l2_overall_metrics.to_csv(
    RESULTS_DIR /
    "l2_logistic_oof_metrics.csv",
    index=False
)

l2_oof_results = pd.DataFrame({
    "Patient": y.index,
    "Actual_Response": y.values,
    "Predicted_Response": (
        l2_oof_prediction_int.values
    ),
    "pCR_Probability": (
        l2_oof_probability.values
    )
})

l2_oof_results.to_csv(
    RESULTS_DIR /
    "l2_logistic_out_of_fold_predictions.csv",
    index=False
)

print("L2 Logistic Regression results saved in:")
print(RESULTS_DIR.resolve())

L2 Logistic Regression results saved in:
D:\diplom-project\results


In [14]:
from pathlib import Path

RESULTS_DIR = Path("results")
RESULTS_DIR.mkdir(
    parents=True,
    exist_ok=True
)

fold_assignment = pd.Series(
    index=X.index,
    dtype="int64",
    name="validation_fold"
)

for fold, (
    training_indices,
    validation_indices
) in enumerate(
    outer_splits,
    start=1
):
    fold_assignment.iloc[
        validation_indices
    ] = fold

fold_assignment_table = (
    fold_assignment
    .rename_axis("Patient")
    .reset_index()
)

assert len(fold_assignment_table) == 306
assert fold_assignment_table["Patient"].is_unique
assert fold_assignment_table["validation_fold"].notna().all()

fold_assignment_table.to_csv(
    RESULTS_DIR /
    "GSE25055_outer_fold_assignments.csv",
    index=False
)

display(fold_assignment_table.head(20))

print("Fold assignments saved.")

,Patient,validation_fold
0,GSM615096,6.0
1,GSM615097,1.0
2,GSM615098,4.0
3,GSM615099,8.0
4,GSM615100,3.0
5,GSM615101,5.0
6,GSM615102,8.0
7,GSM615103,8.0
8,GSM615104,8.0
9,GSM615106,1.0


Fold assignments saved.


In [15]:
fold_assignment_table[
    "validation_fold"
] = (
    fold_assignment_table[
        "validation_fold"
    ]
    .astype("int64")
)

fold_assignment_table.to_csv(
    RESULTS_DIR /
    "GSE25055_outer_fold_assignments.csv",
    index=False
)

display(fold_assignment_table.head(20))

,Patient,validation_fold
0,GSM615096,6
1,GSM615097,1
2,GSM615098,4
3,GSM615099,8
4,GSM615100,3
5,GSM615101,5
6,GSM615102,8
7,GSM615103,8
8,GSM615104,8
9,GSM615106,1


In [16]:
lasso_oof = pd.read_csv(
    RESULTS_DIR /
    "lasso_out_of_fold_predictions.csv"
)

l2_oof = pd.read_csv(
    RESULTS_DIR /
    "l2_logistic_out_of_fold_predictions.csv"
)

lasso_oof = (
    lasso_oof
    .sort_values("Patient")
    .reset_index(drop=True)
)

l2_oof = (
    l2_oof
    .sort_values("Patient")
    .reset_index(drop=True)
)

assert len(lasso_oof) == 306
assert len(l2_oof) == 306

assert lasso_oof["Patient"].equals(
    l2_oof["Patient"]
)

assert lasso_oof["Actual_Response"].equals(
    l2_oof["Actual_Response"]
)

print(
    "The two models contain the same "
    "306 patients and labels."
)

The two models contain the same 306 patients and labels.


In [17]:
def calculate_model_metrics(
    model_name,
    oof_data
):
    y_true = (
        oof_data["Actual_Response"]
        .astype("int64")
    )

    y_predicted = (
        oof_data["Predicted_Response"]
        .astype("int64")
    )

    y_probability = (
        oof_data["pCR_Probability"]
        .astype("float64")
    )

    tn, fp, fn, tp = confusion_matrix(
        y_true,
        y_predicted,
        labels=[0, 1]
    ).ravel()

    sensitivity = (
        tp / (tp + fn)
        if (tp + fn) > 0
        else np.nan
    )

    specificity = (
        tn / (tn + fp)
        if (tn + fp) > 0
        else np.nan
    )

    return {
        "Model": model_name,

        "ROC-AUC": roc_auc_score(
            y_true,
            y_probability
        ),

        "PR-AUC": average_precision_score(
            y_true,
            y_probability
        ),

        "Accuracy": accuracy_score(
            y_true,
            y_predicted
        ),

        "Balanced accuracy": (
            balanced_accuracy_score(
                y_true,
                y_predicted
            )
        ),

        "F1": f1_score(
            y_true,
            y_predicted,
            zero_division=0
        ),

        "Precision": precision_score(
            y_true,
            y_predicted,
            zero_division=0
        ),

        "Sensitivity": sensitivity,
        "Specificity": specificity,

        "True negatives": int(tn),
        "False positives": int(fp),
        "False negatives": int(fn),
        "True positives": int(tp)
    }

In [18]:
model_comparison = pd.DataFrame([
    calculate_model_metrics(
        "LASSO Logistic Regression",
        lasso_oof
    ),
    calculate_model_metrics(
        "L2 Logistic Regression",
        l2_oof
    )
])

display(
    model_comparison.round(4)
)

,Model,ROC-AUC,PR-AUC,Accuracy,Balanced accuracy,F1,Precision,Sensitivity,Specificity,True negatives,False positives,False negatives,True positives
0,LASSO Logistic Regression,0.7268,0.4076,0.8007,0.6002,0.3441,0.4444,0.2807,0.9197,229,20,41,16
1,L2 Logistic Regression,0.7709,0.4143,0.6895,0.7348,0.4920,0.3538,0.8070,0.6627,165,84,11,46


In [19]:
model_comparison.to_csv(
    RESULTS_DIR /
    "model_comparison.csv",
    index=False
)

print("Comparison saved:")
print(
    (
        RESULTS_DIR /
        "model_comparison.csv"
    ).resolve()
)

Comparison saved:
D:\diplom-project\results\model_comparison.csv


In [20]:
from sklearn.ensemble import RandomForestClassifier


def create_library_rf_pipeline():
    return Pipeline([
        (
            "random_forest",
            RandomForestClassifier(
                max_features="sqrt",
                random_state=RANDOM_STATE,
                n_jobs=-1
            )
        )
    ])


rf_parameter_grid = {
    "random_forest__n_estimators": [
        100,
        300
    ],
    "random_forest__max_depth": [
        5,
        None
    ],
    "random_forest__class_weight": [
        None,
        "balanced"
    ]
}

print("Parameter combinations:", 8)
print(create_library_rf_pipeline())

Parameter combinations: 8
Pipeline(steps=[('random_forest',
                 RandomForestClassifier(n_jobs=-1, random_state=42))])


In [21]:
rf_fold_records = []

rf_oof_probability = pd.Series(
    np.nan,
    index=y.index,
    name="pCR_probability"
)

rf_oof_prediction = pd.Series(
    pd.NA,
    index=y.index,
    dtype="Int64",
    name="predicted_response"
)

for fold, (
    training_indices,
    validation_indices
) in enumerate(
    outer_splits,
    start=1
):

    X_train = X.iloc[training_indices]
    X_validation = X.iloc[validation_indices]

    y_train = y.iloc[training_indices]
    y_validation = y.iloc[validation_indices]

    grid_search = GridSearchCV(
        estimator=create_library_rf_pipeline(),
        param_grid=rf_parameter_grid,
        scoring="average_precision",
        cv=inner_cv,
        refit=True,

        # RF itself already uses all processor cores
        n_jobs=1
    )

    grid_search.fit(
        X_train,
        y_train
    )

    best_model = grid_search.best_estimator_

    validation_probability = (
        best_model.predict_proba(
            X_validation
        )[:, 1]
    )

    validation_prediction = (
        best_model.predict(
            X_validation
        )
    )

    rf_oof_probability.loc[
        X_validation.index
    ] = validation_probability

    rf_oof_prediction.loc[
        X_validation.index
    ] = validation_prediction

    tn, fp, fn, tp = confusion_matrix(
        y_validation,
        validation_prediction,
        labels=[0, 1]
    ).ravel()

    sensitivity = (
        tp / (tp + fn)
        if (tp + fn) > 0
        else np.nan
    )

    specificity = (
        tn / (tn + fp)
        if (tn + fp) > 0
        else np.nan
    )

    fold_result = {
        "fold": fold,
        "training_patients": len(training_indices),
        "validation_patients": len(validation_indices),

        "best_n_estimators": (
            grid_search.best_params_[
                "random_forest__n_estimators"
            ]
        ),

        "best_max_depth": (
            grid_search.best_params_[
                "random_forest__max_depth"
            ]
        ),

        "best_class_weight": (
            grid_search.best_params_[
                "random_forest__class_weight"
            ]
        ),

        "roc_auc": roc_auc_score(
            y_validation,
            validation_probability
        ),

        "average_precision": average_precision_score(
            y_validation,
            validation_probability
        ),

        "accuracy": accuracy_score(
            y_validation,
            validation_prediction
        ),

        "balanced_accuracy": balanced_accuracy_score(
            y_validation,
            validation_prediction
        ),

        "f1": f1_score(
            y_validation,
            validation_prediction,
            zero_division=0
        ),

        "precision": precision_score(
            y_validation,
            validation_prediction,
            zero_division=0
        ),

        "sensitivity": sensitivity,
        "specificity": specificity
    }

    rf_fold_records.append(
        fold_result
    )

    print(
        f"Fold {fold}/10 completed | "
        f"trees="
        f"{fold_result['best_n_estimators']} | "
        f"depth="
        f"{fold_result['best_max_depth']} | "
        f"class_weight="
        f"{fold_result['best_class_weight']} | "
        f"ROC-AUC="
        f"{fold_result['roc_auc']:.3f} | "
        f"PR-AUC="
        f"{fold_result['average_precision']:.3f}"
    )

Fold 1/10 completed | trees=300 | depth=None | class_weight=balanced | ROC-AUC=0.847 | PR-AUC=0.655
Fold 2/10 completed | trees=300 | depth=5 | class_weight=balanced | ROC-AUC=0.780 | PR-AUC=0.536
Fold 3/10 completed | trees=100 | depth=5 | class_weight=None | ROC-AUC=0.667 | PR-AUC=0.359
Fold 4/10 completed | trees=100 | depth=None | class_weight=balanced | ROC-AUC=0.640 | PR-AUC=0.273
Fold 5/10 completed | trees=300 | depth=None | class_weight=balanced | ROC-AUC=0.853 | PR-AUC=0.661
Fold 6/10 completed | trees=300 | depth=None | class_weight=None | ROC-AUC=0.690 | PR-AUC=0.482
Fold 7/10 completed | trees=300 | depth=None | class_weight=balanced | ROC-AUC=0.852 | PR-AUC=0.426
Fold 8/10 completed | trees=300 | depth=None | class_weight=balanced | ROC-AUC=0.712 | PR-AUC=0.583
Fold 9/10 completed | trees=100 | depth=5 | class_weight=None | ROC-AUC=0.920 | PR-AUC=0.725
Fold 10/10 completed | trees=300 | depth=None | class_weight=balanced | ROC-AUC=0.729 | PR-AUC=0.377


In [22]:
assert rf_oof_probability.notna().all()
assert rf_oof_prediction.notna().all()

rf_oof_prediction_int = (
    rf_oof_prediction
    .astype("int64")
)

rf_cv_results = pd.DataFrame(
    rf_fold_records
)

display(rf_cv_results)

rf_metric_columns = [
    "roc_auc",
    "average_precision",
    "accuracy",
    "balanced_accuracy",
    "f1",
    "precision",
    "sensitivity",
    "specificity"
]

rf_cv_summary = (
    rf_cv_results[
        rf_metric_columns
    ]
    .agg([
        "mean",
        "std"
    ])
    .T
)

rf_cv_summary.columns = [
    "Mean",
    "Standard deviation"
]

display(rf_cv_summary)


,fold,training_patients,validation_patients,best_n_estimators,best_max_depth,best_class_weight,roc_auc,average_precision,accuracy,balanced_accuracy,f1,precision,sensitivity,specificity
0,1,275,31,300,NaN,balanced,0.846667,0.654762,0.838710,0.646667,0.444444,0.666667,0.333333,0.960000
1,2,275,31,300,5.0,balanced,0.780000,0.535903,0.806452,0.626667,0.400000,0.500000,0.333333,0.920000
2,3,275,31,100,5.0,NaN,0.666667,0.358788,0.806452,0.500000,0.000000,0.000000,0.000000,1.000000
3,4,275,31,100,NaN,balanced,0.640000,0.273357,0.741935,0.460000,0.000000,0.000000,0.000000,0.920000
4,5,275,31,300,NaN,balanced,0.853333,0.660531,0.806452,0.690000,0.500000,0.500000,0.500000,0.880000
5,6,275,31,300,NaN,NaN,0.690000,0.482377,0.806452,0.500000,0.000000,0.000000,0.000000,1.000000
6,7,276,30,300,NaN,balanced,0.852000,0.426465,0.800000,0.800000,0.571429,0.444444,0.800000,0.800000
7,8,276,30,300,NaN,balanced,0.712000,0.582967,0.900000,0.700000,0.571429,1.000000,0.400000,1.000000
8,9,276,30,100,5.0,NaN,0.920000,0.725397,0.866667,0.600000,0.333333,1.000000,0.200000,1.000000
9,10,276,30,300,NaN,balanced,0.729167,0.376894,0.733333,0.583333,0.333333,0.333333,0.333333,0.833333


,Mean,Standard deviation
roc_auc,0.768983,0.094953
average_precision,0.507744,0.149175
accuracy,0.810645,0.050324
balanced_accuracy,0.610667,0.105266
f1,0.315397,0.232872
precision,0.444444,0.376796
sensitivity,0.290000,0.254369
specificity,0.931333,0.074107


In [23]:
rf_confusion_matrix = confusion_matrix(
    y,
    rf_oof_prediction_int,
    labels=[0, 1]
)

tn, fp, fn, tp = (
    rf_confusion_matrix.ravel()
)

rf_sensitivity = (
    tp / (tp + fn)
)

rf_specificity = (
    tn / (tn + fp)
)

rf_overall_metrics = pd.DataFrame({
    "Metric": [
        "ROC-AUC",
        "PR-AUC",
        "Accuracy",
        "Balanced accuracy",
        "F1",
        "Precision",
        "Sensitivity",
        "Specificity"
    ],
    "Value": [
        roc_auc_score(
            y,
            rf_oof_probability
        ),
        average_precision_score(
            y,
            rf_oof_probability
        ),
        accuracy_score(
            y,
            rf_oof_prediction_int
        ),
        balanced_accuracy_score(
            y,
            rf_oof_prediction_int
        ),
        f1_score(
            y,
            rf_oof_prediction_int,
            zero_division=0
        ),
        precision_score(
            y,
            rf_oof_prediction_int,
            zero_division=0
        ),
        rf_sensitivity,
        rf_specificity
    ]
})

display(rf_overall_metrics)

print("\nConfusion matrix:")
print(rf_confusion_matrix)



,Metric,Value
0,ROC-AUC,0.760445
1,PR-AUC,0.408074
2,Accuracy,0.810458
3,Balanced accuracy,0.606214
4,F1,0.355556
5,Precision,0.484848
6,Sensitivity,0.280702
7,Specificity,0.931727



Confusion matrix:
[[232  17]
 [ 41  16]]


In [24]:
rf_cv_results.to_csv(
    RESULTS_DIR /
    "library_rf_cv_fold_metrics.csv",
    index=False
)

rf_cv_summary.to_csv(
    RESULTS_DIR /
    "library_rf_cv_summary.csv"
)

rf_overall_metrics.to_csv(
    RESULTS_DIR /
    "library_rf_oof_metrics.csv",
    index=False
)

rf_oof_results = pd.DataFrame({
    "Patient": y.index,
    "Actual_Response": y.values,
    "Predicted_Response": (
        rf_oof_prediction_int.values
    ),
    "pCR_Probability": (
        rf_oof_probability.values
    ),
    "Validation_Fold": (
        fold_assignment_table[
            "validation_fold"
        ].values
    )
})

rf_oof_results.to_csv(
    RESULTS_DIR /
    "library_rf_out_of_fold_predictions.csv",
    index=False
)

print("Library Random Forest results saved.")


Library Random Forest results saved.
